# Risk Across Tails and Timescales
### SAIFA Quant Edge 1.0, Round 1 · Team Gmora

**Research question.** Does tail dependence between US equity sectors change with the investment horizon, and what does
ignoring this do to a portfolio's measured risk?

This notebook is the complete, connected analysis behind our report. It runs from raw data to the final
recommendation in one chain:

1. Data loading and quality checks
2. Visualisation
3. Exploratory data analysis
4. Preprocessing (GARCH filtering, wavelet decomposition, pseudo-observations)
5. Model building: tail dependence across horizons (part a of the question)
6. Out-of-sample forecasting engine
7. Evaluation: what ignoring tail dependence does to measured risk (part b)
8. Recommendation, robustness and the report PDF

At each step we state what we see and the decision we take before moving on. All heavy computation lives in the
`src/` package so that this notebook, `python reproduce.py` and the report produce identical numbers.

**How to run.** Open this notebook from the unzipped project folder and choose *Run All*. Everything runs locally:
the prices are bundled in `data/etf_prices.csv`, so nothing is downloaded apart from the Python packages in the
first cell. A full run takes about 10 minutes on a laptop (Python 3.10 to 3.13).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sys, json, runpy, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / "src" / "run_all.py").exists(), "Open this notebook from the project folder (the one that contains src/)."
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

from src import config, eda, figures
from src import run_all as pipeline
ASSETS = ROOT / "report" / "assets"
def show(name, width=900):
    display(Image(filename=str(ASSETS / name), width=width))
print("Python", sys.version.split()[0])

Before any analysis we confirm that the statistical building blocks behave as intended. The unit tests check,
among other things, that copula selection by AIC recovers the true family on simulated data, that the wavelet
decomposition adds back exactly to the original series, and that every backtest statistic matches known values.

In [ ]:
import pytest
assert pytest.main(["-q", "tests"]) == 0, "unit tests failed"

## 1. Data loading and quality checks

**Portfolio.** An equal-weight portfolio of six original Select Sector SPDR ETFs: energy (XLE), financials (XLF),
technology (XLK), health care (XLV), industrials (XLI) and utilities (XLU). Sectors are where an equity investor
expects diversification to come from, so they are the natural place to ask whether diversification survives
crashes and longer holding periods. Equal weights keep the result about dependence rather than about weights. The
S&P 500 ETF (SPY) is in the data file but excluded from the portfolio because it contains the sectors and would
double-count market exposure.

**Source.** Yahoo Finance daily adjusted closes (dividends and splits included), 1999 to 2026. The US market gives
deep, liquid, synchronously traded prices across four crises: the dot-com bust, the 2008 financial crisis, COVID and
the 2022 rate shock.

In [ ]:
raw, prices, rets = pipeline.stage_data()
display(prices.head(3)); display(prices.tail(3))
print(f"{len(prices):,} trading days, {prices.shape[1]} ETFs; {len(rets):,} daily log returns")

In [ ]:
quality = eda.data_quality(raw[config.TICKERS], prices)
display(quality)
print("Duplicate dates:", quality.attrs["duplicate_dates"], "| rows dropped when aligning tickers:", quality.attrs["rows_dropped_by_alignment"])

**What we see.** No missing prices, no duplicate dates and no zero or negative prices. Days with an exactly zero
return are rare (about 1 to 1.5% of days per ETF), which is normal for liquid ETFs and not a sign of stale data. The
largest one-day moves fall on known market events (for example energy in March 2020 and financials in late 2008).

**Decision.** Use the data as it is: no imputation, no outlier removal. Extreme days are exactly what a tail-risk
study is about, so removing them would bias the answer. We model daily **log returns** (they add over time, which
the multi-day analysis needs) and compute portfolio profit and loss from **simple returns** (they add across assets).

**Sample split.** All model choices and the horizon analysis use 1999 to 2019 only. Everything from 1 January 2020
onward is held out for out-of-sample testing, so the test period contains a crash (COVID) and a rate shock (2022)
that the models never saw.

## 2. Visualisation

In [ ]:
avg_corr = eda.rolling_avg_corr(rets)
figures.fig_data_overview(rets, avg_corr, config.OOS_START)
show("fig0a_data_overview.png")
vol = (np.expm1(rets).mean(axis=1).rolling(250).std() * np.sqrt(252)).reindex(avg_corr.index)
ok = avg_corr.notna() & vol.notna()
print(f"Average pairwise correlation ranges from {avg_corr.min():.2f} to {avg_corr.max():.2f}; "
      f"its correlation with portfolio volatility is {np.corrcoef(avg_corr[ok], vol[ok])[0, 1]:.2f}")

**What we see.** Panel (a): the sectors follow very different long-run paths (energy and technology swap
leadership), which is the usual argument for sector diversification. Panels (b) and (c): volatility comes in
bursts (2002, 2008 to 2009, 2020, 2022) and the average correlation between sectors rises in the same episodes,
from about 0.2 in calm years to above 0.8 in 2011 to 2012 and 2020. In other words, diversification is weakest
exactly when it is needed.

**Decision.** Two features have to be in the model: (1) time-varying volatility, and (2) dependence that is
stronger in turbulent markets than a single correlation number suggests. We also re-estimate the models on a
rolling window, because the level of correlation drifts over time.

## 3. Exploratory data analysis

We now test formally what the pictures suggest, on the 1999 to 2019 estimation sample.

In [ ]:
ris = rets.loc[:config.IN_SAMPLE_END]
port = np.log1p(np.expm1(ris).mean(axis=1)).rename("Portfolio")
stats_tbl = eda.return_stats(ris.join(port))
display(stats_tbl)

**What we see.** Annual volatility ranges from about 18% (health care, utilities) to 29% (financials). Every series
has large excess kurtosis (between about 5.6 and 14.5; a normal distribution has 0) and the Jarque–Bera test rejects
normality for all of them. The Ljung–Box test on squared returns and Engle's ARCH-LM test reject with p-values
near zero for every sector: large moves are followed by large moves.

**Decision.** Use **GARCH(1,1) with Student-t innovations** for each sector: GARCH captures the volatility clustering
and the t distribution the fat tails. A normal model with constant volatility would understate tail losses.

In [ ]:
display(Markdown("**Correlation matrix of daily log returns, 1999 to 2019**"))
display(ris.corr().round(2))
display(Markdown("**Do sectors hit extremes together?** On days when one sector is in its worst (or best) q-tail, "
                 "how many of the other five are too?"))
display(eda.co_extremes(ris))

**What we see.** Pairwise correlations run from about 0.43 (technology and utilities) to 0.76 (financials and
industrials). More telling for risk: on a day when one sector has a worst-5% day, on average about 2.1 of the other
five sectors do too. If sectors were independent the figure would be 0.25. All six sectors were in their worst 5%
together on 33 days. Extremes cluster across sectors.

**Decision.** A correlation matrix is not enough to describe this. We need copulas, which separate each sector's
own distribution from the dependence between sectors, and measures of **tail dependence**, which ask directly how
likely sectors are to crash together. Because the question is about horizons, we will measure tail dependence at
several time scales.

## 4. Preprocessing

Three steps turn returns into inputs for the dependence models.

**Step 1: GARCH filtering.** Fit GARCH(1,1)-t to each sector and keep the standardised residuals
z = (r − μ) / σ. These are the shocks after removing each sector's changing volatility. Dependence must be measured
on z, not on raw returns: otherwise two sectors look tail-dependent simply because both are volatile at the same
time.

In [ ]:
fits_is = pipeline.fit_all_margins(ris)
z = pipeline.std_resid_matrix(fits_is)
display(eda.garch_diagnostics(fits_is))
figures.fig_diagnostics(port, ris, z, eda.acf)
show("fig0b_diagnostics.png")

**What we see.** Persistence α + β is between 0.987 and 0.999 for all sectors: volatility shocks die out slowly,
with half-lives of roughly two months to two years. The estimated t degrees of freedom (6 to 12) confirm fat tails
even after allowing for changing volatility. After filtering, the autocorrelation of squared residuals falls inside
the no-autocorrelation band (right panel), excess kurtosis drops from 5.6 to 14.5 to between 0.7 and 2.2, and the
Ljung–Box test on squared residuals no longer rejects for five of six sectors. Utilities keep mild residual
clustering, which we note as a limitation.

**Decision.** The GARCH-t filter does its job. From here on, dependence is measured on the standardised residuals.

**Step 2: wavelet decomposition into horizons.** To ask whether dependence changes with the horizon we split each
residual series into components that live on different time scales. We use the maximal overlap discrete wavelet
transform (MODWT) with the LA(8) filter and six levels. Each level j captures fluctuations lasting about 2^(j−1) to
2^j days. We group them into three horizon bands:

| Band | Levels | Time scale | Interpretation |
|---|---|---|---|
| H1 | D1 + D2 | 2 to 8 days | trading horizon |
| H2 | D3 + D4 | 8 to 32 days | weeks to a month; contains the 10-day regulatory horizon |
| H3 | D5 + D6 + smooth | more than 32 days | quarterly, strategic horizon |

We compute the multiresolution analysis in the frequency domain so that the three bands add back to the series
exactly (a unit test checks this), reflect the series at its ends, and drop boundary-affected values in the
in-sample analysis. The example below shows financials during 2007 to 2009.

In [ ]:
from src.wavelets import decompose
bands_xlf = {b: pd.Series(v[:, 0], index=z.index) for b, v in decompose(z[["XLF"]], trim=False).items()}
figures.fig_wavelet_example(z["XLF"], bands_xlf, "2007-01-01", "2009-12-31")
show("fig0c_wavelet_bands.png")
display(Markdown("**Share of each sector's residual variance in each band**"))
display(eda.wavelet_energy(z))

**What we see.** H1 keeps the day-to-day noise, H2 the swings over a few weeks, H3 the slow drifts over months.
Roughly 79% of the residual variance sits in H1, 16% in H2 and 5% in H3, close to the 75/19/6 split of pure white
noise. This is expected: GARCH has already removed the predictable part of volatility. It also means the bands differ
in **how sectors move together**, not in how much each sector moves, which is exactly what we want to study.

**Step 3: pseudo-observations.** Before fitting copulas we replace each series by its ranks divided by (n + 1). This
makes the dependence estimates independent of the marginal distributions (the canonical maximum-likelihood approach of
Genest, Ghoudi and Rivest, 1995).

## 5. Model building: does tail dependence change with the horizon?

**5.1 Which copula family fits best at each horizon?** We fit three copulas to the daily residuals and to each band
by maximum likelihood with the full six-dimensional density, and compare them by AIC:

* **Gaussian**: no tail dependence, the implicit assumption of a correlation-matrix risk model;
* **Student-t**: symmetric tail dependence (crashes and rallies equally joint);
* **Clayton**: dependence in the crash tail only.

This step and the next take about two to three minutes because of the bootstrap.

In [ ]:
hz = pipeline.stage_horizon(ris, fits_is, z)
cop = hz["cop"]
display(cop[["series", "family", "loglik", "n_params", "aic", "delta_aic", "aic_winner", "nu", "lambda_l"]])

**What we see.** The Student-t copula wins at every scale by a large AIC margin (hundreds to thousands of points),
so tail dependence is real at all horizons. The single-parameter Clayton copula loses badly because it forces every
pair of sectors to have the same dependence. The t-copula's tail-dependence coefficient λ is about 0.14 to 0.15
from daily data up to the 8 to 32 day band, then drops to 0.05 beyond 32 days (with ν rising to 16).

Taken at face value, this says tail dependence fades at long horizons. But the t-copula forces the crash tail and the
rally tail to be equal. **Decision:** measure the two tails separately, without imposing any parametric shape.

**5.2 Crash and rally co-movement by horizon.** For each pair of sectors we estimate
χ_L(q) = P(sector j in its worst q-tail | sector i in its worst q-tail), the probability of crashing together, and
χ_U(q), the same for the best tail. We average over the 15 pairs and compare with the value a Gaussian copula with
the same correlation would imply. Confidence intervals and tests come from a stationary block bootstrap (400
replications, mean block length 60 days) that re-runs the wavelet decomposition on every resample.

In [ ]:
show("fig1_tail_by_horizon.png")
lv = hz["band_lv"]
display(lv[lv.q == 0.05][["series", "scale", "tail", "est", "lo", "hi", "gaussian_implied", "excess_est", "excess_p0"]])
ts = hz["band_ts"]
display(ts[ts.q == 0.05][["test", "est", "lo", "hi", "p0"]].rename(columns={"p0": "p_value"}))

**What we see.**

* **Crash co-movement does not fade.** χ_L is 0.37 at the 2 to 8 day scale and 0.31 beyond 32 days. The difference
  (−0.05) is not statistically significant (p ≈ 0.21).
* **Rally co-movement does fade.** χ_U falls from 0.35 to 0.20, a highly significant drop (p < 0.001).
* **So dependence becomes asymmetric at long horizons.** At short scales the two tails are equal; beyond a month crash
  co-movement exceeds rally co-movement by 0.11 (p ≈ 0.03). Sectors rally separately but fall together, and this
  does not wash out as the holding period lengthens.
* **Crash co-movement exceeds what a Gaussian copula implies** at every scale (by about 0.05 to 0.10).

This explains the t-copula result in 5.1: a symmetric model averages a persistent crash tail with a fading rally tail
and reports that tail dependence fades. It would therefore **understate long-horizon crash risk**.

Panel (b) of the figure repeats the analysis on raw, non-overlapping 1, 5, 10 and 21-day returns. With only 251
independent 21-day observations the intervals are wide, but crash co-movement stays above the Gaussian level up to
10 days.

**5.3 Can a model estimated on daily data reproduce this?** Risk systems usually estimate a daily model and
simulate it forward to longer horizons. We simulate 60,000 paths of each daily model (GARCH margins plus Gaussian,
t or empirical copula) and compare the crash co-movement they imply at each horizon with what was observed.

In [ ]:
obs = hz["hor_lv"]
obs = obs[(obs.q == 0.10) & (obs["tail"] == "L")].set_index("horizon_days")[["est", "lo", "hi"]]
imp = hz["implied"]
imp = imp[imp.q == 0.10].pivot(index="horizon_days", columns="model", values="chi_L")
display(obs.rename(columns={"est": "observed", "lo": "CI low", "hi": "CI high"}).join(imp))

**What we see.** At one day the Gaussian model implies 0.38 and the t-copula 0.40, both below the 95% interval of
the observed 0.46; only the empirical copula (0.44) is inside. At longer horizons every model stays below the observed
value, though the intervals widen.

**Decisions for the forecasting models.**
1. Do not use a Gaussian copula for risk: it misses the crash tail at every horizon.
2. Compare a t-copula and an empirical (non-parametric) copula, which keeps any asymmetry.
3. Because crash co-movement at 8 to 32 days is statistically the same as at the daily scale, daily crash
   dependence can legitimately be carried to a 10-day horizon. We test whether imposing it directly at 10 days,
   rather than compounding independent daily draws, changes measured risk.

## 6. Out-of-sample forecasting engine

Every forecast for day t uses only data up to day t − 1. Each 20 trading days we re-estimate GARCH and the copulas on
the trailing 1,000 days (about four years); between re-estimations the GARCH volatility is updated every day with the
latest return. Shocks are drawn from each sector's own empirical residual distribution (filtered historical
simulation), joined by the copula, and scaled by today's volatility.

**One-day models** (same margins, different dependence, plus a model-free benchmark):

| Code | Model | Role |
|---|---|---|
| HS | historical simulation, last 250 days | industry default, simple benchmark |
| FHS | empirical copula of daily residuals (filtered HS) | non-parametric dependence |
| G | Gaussian copula | benchmark that ignores tail dependence |
| T | Student-t copula | symmetric tail dependence |
| WT | t-copula fitted on the H1 wavelet band | tail dependence at the 2 to 8 day scale |

**Ten-day models** on 169 non-overlapping windows change one simplification at a time:

| Code | Model | What it ignores |
|---|---|---|
| G_sqrt | Gaussian one-day risk × √10 | tail dependence; uses the square-root-of-time rule |
| G_path | Gaussian copula, simulated 10-day GARCH paths | tail dependence |
| T_path | t-copula, simulated paths | compounds independent daily draws |
| T_join | daily t-copula imposed directly at 10 days | asymmetry |
| WC | t-copula fitted on the H2 band | asymmetry (symmetric t) |
| E_join | daily empirical copula imposed at 10 days | uses daily rather than band data |
| WC_emp | empirical copula of the H2 band | nothing: the full horizon-aware model |

This step takes about 1.5 minutes.

In [ ]:
oo = pipeline.stage_oos(rets)
d1, dh, rf = oo["d1"], oo["dh"], oo["rf"]
print(f"{len(d1):,} one-day forecasts, {len(dh)} ten-day windows, {len(rf)} re-estimations")
figures.fig_refits(rf)
show("fig5_rolling_lambda.png")

**What we see.** The rolling t-copula tail dependence rises in stress episodes (2020, 2022) at every scale, and the
8 to 32 day band is the most variable. Dependence is time-varying, which supports re-estimating it regularly rather
than fixing it once.

## 7. Evaluation: what does ignoring tail dependence do to measured risk?

We backtest every model with standard regulatory and academic tests:

* **VaR:** Kupiec coverage test (right number of breaches?), Christoffersen independence test (do breaches cluster?),
  Engle–Manganelli dynamic quantile test, and the Basel traffic light (rolling 250-day count of 99% breaches).
* **Expected Shortfall:** Acerbi–Szekely Z2 test and McNeil–Frey exceedance test (are losses beyond VaR as large as
  predicted?).
* **Model comparison:** the FZ0 joint VaR–ES scoring function with Diebold–Mariano tests.
* **Monte Carlo error:** the whole out-of-sample engine is re-run with three more random seeds.

This step takes about four minutes because of the seed re-runs.

In [ ]:
eda_summary = {"duplicate_dates": quality.attrs["duplicate_dates"],
               "rows_dropped_by_alignment": quality.attrs["rows_dropped_by_alignment"],
               "corr_vol_link": float(np.corrcoef(avg_corr[ok], vol[ok])[0, 1]),
               "avg_corr_min": float(avg_corr.min()), "avg_corr_max": float(avg_corr.max())}
for k, df in {"quality": quality, "stats": stats_tbl, "corr": ris.corr(), "co_extremes": eda.co_extremes(ris),
              "garch": eda.garch_diagnostics(fits_is), "energy": eda.wavelet_energy(z)}.items():
    df.to_csv(ROOT / "outputs" / f"eda_{k}.csv")
ev = pipeline.stage_evaluate(prices, rets, hz, oo, eda_summary)
H = ev["H"]
rows = []
for m, name in [("HS", "Historical simulation"), ("FHS", "Filtered HS"), ("G", "Gaussian copula"),
                ("T", "Student-t copula"), ("WT", "Wavelet t-copula (H1)")]:
    s = H["oos_1d"][m]
    rows.append({"model": name, "99% breaches": s["var99"]["hits"], "expected": round(s["var99"]["expected_hits"]),
                 "Kupiec p": s["var99"]["p_uc"], "Christoffersen p": s["var99"]["p_ind"], "DQ p": s["var99"]["p_dq"],
                 "Basel red share": s["traffic_light"]["red_share"], "mean ES 97.5%": s["mean_es97_5"],
                 "Z2 p": s["es97_5"]["p_z2"], "McNeil-Frey p": s["es97_5"]["mf_p"], "FZ0 loss": s["fz0_97_5"]})
display(Markdown("**One-day risk, out-of-sample 2020 to 2026**"))
display(pd.DataFrame(rows).set_index("model"))
display(Markdown("**Diebold–Mariano tests on the FZ0 loss** (negative statistic: first model better)"))
display(pd.DataFrame(H["oos_1d"]["dm_fz0_97_5"]).T)
show("fig2_oos_var99.png"); show("fig3_traffic_light.png")
display(Markdown("**Breach rates (%) by sub-period**"))
display(ev["sub"].pivot(index="period", columns="model", values="hit99_pct"))

**What we see at one day.**

* The **Gaussian copula breaches its 99% VaR 21 times against 17 expected**, in every Monte Carlo seed, and is the
  only copula model to enter the Basel red zone. Its average ES is about 4% below the t-copula's and 10% below
  filtered historical simulation.
* The t-copula (17 to 19 breaches) and the wavelet t-copula (15 to 18) are close to target. All copula models pass the
  Kupiec and Acerbi–Szekely tests.
* On the joint FZ0 score the copula models are statistically tied, while all of them beat historical simulation
  decisively. At one day, **getting volatility right matters most** and tail dependence is a second-order,
  tail-only effect. Using the H1 band instead of daily residuals changes little, as section 5 predicts.
* **What still fails:** breaches cluster during COVID, so the dynamic-quantile test rejects every model at 99%, and
  realised losses on breach days are 10 to 16% larger than predicted ES. A model re-estimated monthly on four years of
  data cannot anticipate a shock the size of March 2020.

In [ ]:
rows = []
for m in ["G_sqrt", "G_path", "T_path", "T_join", "WC", "E_join", "WC_emp"]:
    s = H["oos_10d"][m]
    rel = dh[f"es97_5_{m}"] / dh["es97_5_WC_emp"] - 1
    stressed = dh["es97_5_WC_emp"] >= dh["es97_5_WC_emp"].quantile(0.75)
    rows.append({"model": m, "mean ES 97.5%": s["mean_es97_5"], "gap vs full model": rel.mean(),
                 "gap when stressed": rel[stressed].mean(), "99% breaches": s["var99"]["hits"],
                 "95% breaches": s["var95"]["hits"], "breach loss / ES": s["es97_5"]["realised_over_predicted"],
                 "McNeil-Frey p": s["es97_5"]["mf_p"], "FZ0 loss": s["fz0_97_5"]})
display(Markdown("**Ten-day risk, 169 non-overlapping windows, 2020 to 2026**"))
display(pd.DataFrame(rows).set_index("model"))
show("fig4_es10d.png")

**What we see at ten days.** This is where the horizon matters.

1. **Ignoring tail dependence** (Gaussian paths) puts 10-day ES about 12% below the full model on average and about
   15% below when risk is high. This model also has the largest losses on breach days relative to its ES.
2. **Compounding daily draws** of a t-copula still leaves about 10%: independent daily draws dilute the joint crash
   tail, while section 5 showed the observed crash tail does not fade. Imposing the daily copula directly at the
   10-day horizon removes most of that gap.
3. **The shape of the tail matters.** The empirical copula, which keeps the crash/rally asymmetry, adds a further
   7 to 8% over the symmetric t-copula.
4. **Daily versus band data.** Estimating the copula on the H2 band gives almost the same answer as daily data
   (within about 1%), which is what section 5 predicts: crash co-movement at 8 to 32 days equals the daily level.
   The wavelet analysis is what justifies carrying daily crash dependence to the 10-day horizon.
5. **The √10 rule** lands about 6.5% below the full model. It does better than Gaussian paths only because two errors
   partly cancel: it ignores tail dependence, but scaling a fat-tailed one-day distribution by √10 overstates the
   10-day tail.
6. With 169 windows the backtest has little power: no model's breach count is rejected and the FZ0 differences are not
   significant. The two path-compounding models are the only ones whose breach losses are significantly larger than
   their predicted ES.

In [ ]:
rob = pd.read_csv(ROOT / "outputs" / "seed_robustness.csv")
display(Markdown("**Monte Carlo seed robustness: 99% breaches per model** (main run and three more seeds)"))
display(rob[["seed"] + [c for c in rob.columns if c.startswith("hits99_")]])

**What we see.** Breach counts move by at most a few across seeds and the ranking of models does not change, so
the conclusions are not an artefact of Monte Carlo noise.

## 8. Answer and recommendation

**(a) Does tail dependence change with the horizon?** Partly. Crash co-movement between US sectors does not weaken as
the horizon lengthens from days to months, while rally co-movement roughly halves. Dependence therefore becomes
asymmetric at long horizons, and at every horizon crash co-movement is higher than a Gaussian copula implies.

**(b) What does ignoring it do to measured risk?** It understates risk, modestly at one day and materially at ten
days: about 4% of daily ES and 21 instead of 17 breaches at 99% for a Gaussian copula, and about 12% of 10-day ES
(15% in stressed periods) for a Gaussian path model.

The desk recommendation below is generated from the results:

In [ ]:
print(ev["rec"])

## 9. Report

The final cell rebuilds the PDF report from the outputs just produced. Every number in the report is read from
`outputs/results.json` and the CSV files in `outputs/`; none is typed by hand.

In [ ]:
runpy.run_path(str(ROOT / "scripts" / "build_report_pdf.py"), run_name="__main__")
print("Outputs:", sorted(p.name for p in (ROOT / "outputs").glob("*") if p.suffix in {".csv", ".json", ".txt"}))

## Limitations and further work

* Long-scale bands are highly autocorrelated, so H3 has few effective observations; the bootstrap intervals widen
  accordingly but remain approximate.
* The 10-day backtest has low power (169 windows).
* The t-copula has a single degrees-of-freedom parameter and cannot be asymmetric; the empirical copula captures
  asymmetry but cannot extrapolate beyond the worst event in its window.
* Breach clustering remains; dynamic (DCC or GAS) or regime-switching copulas are the natural next step, together
  with asymmetric parametric copulas per scale (skew-t, vine copulas).
* The study covers one equal-weight US sector portfolio; the code is portfolio-agnostic and could be applied to the
  Colombo Stock Exchange with corrections for non-synchronous trading.